# Preparación del dataset para minería y similaridad

Este notebook prepara los datos que utilizarán los experimentos de reglas de asociación y PSO.

In [ ]:
# Importaciones principales

import glob
import os
from math import erf, sqrt

import kagglehub
import numpy as np
import pandas as pd

## 1. Fuente y objetivo

Se utiliza el **Cardiovascular Disease Dataset**, disponible en Kaggle:

https://www.kaggle.com/datasets/sulianova/cardiovascular-disease-dataset

El objetivo de esta etapa es generar dos representaciones compatibles con las fases posteriores:

- una versión preparada en unidades interpretables;
- una versión normalizada para las funciones de similaridad y el algoritmo PSO.

## 2. Descripción del dataset

El dataset contiene registros de pacientes y variables objetivas, clínicas y de estilo de vida. La variable objetivo indica la presencia o ausencia de enfermedad cardiovascular.

| Columna original | Nombre utilizado | Descripción |
|---|---|---|
| `age` | `edad_dias` | Edad del paciente medida en días. |
| `gender` | `sexo` | 1 = mujer; 2 = hombre. |
| `height` | `altura_cm` | Altura en centímetros. |
| `weight` | `peso_kg` | Peso en kilogramos. |
| `ap_hi` | `presion_sistolica` | Presión arterial sistólica. |
| `ap_lo` | `presion_diastolica` | Presión arterial diastólica. |
| `cholesterol` | `colesterol` | 1 = normal; 2 = arriba de lo normal; 3 = muy arriba de lo normal. |
| `gluc` | `glucosa` | 1 = normal; 2 = arriba de lo normal; 3 = muy arriba de lo normal. |
| `smoke` | `fumador` | 0 = no; 1 = sí. |
| `alco` | `alcohol` | 0 = no; 1 = sí. |
| `active` | `actividad_fisica` | 0 = no; 1 = sí. |
| `cardio` | `enfermedad_cardiovascular` | 0 = ausencia; 1 = presencia. |

La columna `id` se elimina porque funciona únicamente como identificador y no representa una condición clínica.

In [ ]:
# Descargar el dataset desde Kaggle

path = kagglehub.dataset_download(
    "sulianova/cardiovascular-disease-dataset"
)

print(f"Archivos disponibles en: {path}")

In [ ]:
# 3. Cargar y validar el archivo CSV

csv_files = glob.glob(os.path.join(path, "*.csv"))

if not csv_files:
    raise FileNotFoundError(
        f"No se encontraron archivos CSV en: {path}"
    )

ruta_csv = next(
    (
        archivo
        for archivo in csv_files
        if os.path.basename(archivo) == "cardio_train.csv"
    ),
    csv_files[0]
)

df = pd.read_csv(ruta_csv, sep=";")

print(f"Archivo cargado: {ruta_csv}")
print(f"Dimensiones iniciales: {df.shape}")
display(df.head())
df.info()

In [ ]:
# 4. Limpieza y renombrado

df_limpio = df.drop(columns=["id"], errors="ignore").copy()

resumen_nulos = df_limpio.isna().sum()
columnas_con_nulos = resumen_nulos[resumen_nulos > 0]

if not columnas_con_nulos.empty:
    raise ValueError(
        "Se encontraron valores nulos. Deben resolverse explícitamente "
        f"antes de continuar:\n{columnas_con_nulos}"
    )

nombres_columnas = {
    "age": "edad_dias",
    "gender": "sexo",
    "height": "altura_cm",
    "weight": "peso_kg",
    "ap_hi": "presion_sistolica",
    "ap_lo": "presion_diastolica",
    "cholesterol": "colesterol",
    "gluc": "glucosa",
    "smoke": "fumador",
    "alco": "alcohol",
    "active": "actividad_fisica",
    "cardio": "enfermedad_cardiovascular"
}

df_limpio = df_limpio.rename(
    columns=nombres_columnas,
    errors="raise"
)

print(f"Dimensiones después de la limpieza: {df_limpio.shape}")

In [ ]:
## 5. Conversión de edad

df_limpio["edad_anios"] = (
    df_limpio["edad_dias"] / 365.25
).round(0)

df_limpio = df_limpio.drop(columns=["edad_dias"])

display(df_limpio.head())

## 6. Preprocesamiento robusto de variables continuas

Las variables continuas presentan escalas diferentes y pueden contener valores extremos que deformen una normalización Min-Max convencional. Para reducir este efecto sin eliminar observaciones, se utiliza un criterio híbrido que combina cuantiles empíricos y el rango intercuartílico.

Primero se calcula una cobertura equivalente al intervalo de tres desviaciones estándar alrededor de la media de una distribución normal:

$$
P(-3\\sigma \\leq X \\leq 3\\sigma)
=2\\Phi(3)-1
\\approx 0.9973
$$

Por lo tanto, la probabilidad correspondiente a cada extremo es:

$$
\\alpha =
\\frac{1-0.9973}{2}
\\approx 0.00135
$$

Para cada variable continua `Xj`, se calculan los límites empíricos mediante cuantiles:

$$
L_{Q,j}=Q_{\\alpha}(X_j)
$$

$$
U_{Q,j}=Q_{1-\\alpha}(X_j)
$$

También se calcula el rango intercuartílico:

$$
IQR_j=Q_{3,j}-Q_{1,j}
$$

A partir de este rango se definen las cercas exteriores:

$$
L_{IQR,j}=Q_{1,j}-3IQR_j
$$

$$
U_{IQR,j}=Q_{3,j}+3IQR_j
$$

Los límites híbridos se obtienen seleccionando el límite más restrictivo en cada extremo:

$$
L_j=\\max(L_{Q,j},L_{IQR,j})
$$

$$
U_j=\\min(U_{Q,j},U_{IQR,j})
$$

Los registros que quedan fuera de este intervalo no se eliminan. En la copia destinada al algoritmo PSO, sus valores se recortan mediante la función de clipping:

$$
x_{ij}^{*}=
\\begin{cases}
L_j, & \\text{si } x_{ij}<L_j,\\
x_{ij}, & \\text{si } L_j\\leq x_{ij}\\leq U_j,\\
U_j, & \\text{si } x_{ij}>U_j.
\\end{cases}
$$

Finalmente, cada variable continua se transforma al intervalo `[0,1]`:

$$
x_{ij}^{\\mathrm{norm}}
=
\\frac{x_{ij}^{*}-L_j}{U_j-L_j}
$$

Este procedimiento evita que unos pocos valores extremos dominen la escala de las variables, pero mantiene todas las filas del dataset. Los valores originales y los parámetros utilizados para la transformación se conservan por separado, permitiendo interpretar y auditar posteriormente las reglas descubiertas.

In [ ]:
# ============================================================
# NORMALIZACIÃ“N ROBUSTA CON CRITERIO HÃBRIDO:
# CUANTILES DE Â±3 SIGMA + CERCAS IQR
# ============================================================


from math import erf, sqrt


def normalizar_robustamente(
    dataframe,
    columnas_continuas,
    numero_sigmas=3.0,
    factor_iqr=3.0
):
    """
    Normaliza variables continuas mediante un criterio hÃ­brido.

    Para cada variable se calculan dos pares de lÃ­mites:

    1. LÃ­mites por cuantiles equivalentes a Â±numero_sigmas.
    2. Cercas exteriores basadas en el rango intercuartÃ­lico.

    Finalmente se seleccionan los lÃ­mites mÃ¡s conservadores:

        limite_inferior = max(
            limite_cuantil_inferior,
            limite_iqr_inferior
        )

        limite_superior = min(
            limite_cuantil_superior,
            limite_iqr_superior
        )

    Los valores fuera del intervalo se recortan mediante
    clipping. No se eliminan filas.

    ParÃ¡metros
    ----------
    dataframe : pandas.DataFrame
        Dataset en unidades originales.

    columnas_continuas : list
        Variables continuas que serÃ¡n normalizadas.

    numero_sigmas : float, default=3.0
        NÃºmero de sigmas utilizado para calcular la cobertura
        normal equivalente.

    factor_iqr : float, default=3.0
        Multiplicador utilizado en las cercas del IQR.

        Con factor_iqr=3 se utilizan las cercas exteriores
        de Tukey, menos agresivas que las cercas de 1.5 IQR.

    Retorna
    -------
    dataframe_normalizado : pandas.DataFrame
        Copia del dataset con las columnas continuas
        normalizadas en [0, 1].

    parametros_diccionario : dict
        ParÃ¡metros de transformaciÃ³n de cada variable.

    resumen : pandas.DataFrame
        Resumen de lÃ­mites y valores recortados.
    """

    # --------------------------------------------------------
    # 1. Validaciones
    # --------------------------------------------------------

    if not isinstance(dataframe, pd.DataFrame):
        raise TypeError(
            "'dataframe' debe ser un pandas.DataFrame."
        )

    if numero_sigmas <= 0:
        raise ValueError(
            "'numero_sigmas' debe ser mayor que cero."
        )

    if factor_iqr <= 0:
        raise ValueError(
            "'factor_iqr' debe ser mayor que cero."
        )

    columnas_faltantes = [
        columna
        for columna in columnas_continuas
        if columna not in dataframe.columns
    ]

    if columnas_faltantes:
        raise ValueError(
            "No se encontraron las siguientes columnas: "
            f"{columnas_faltantes}"
        )

    columnas_no_numericas = [
        columna
        for columna in columnas_continuas
        if not pd.api.types.is_numeric_dtype(
            dataframe[columna]
        )
    ]

    if columnas_no_numericas:
        raise TypeError(
            "Las siguientes columnas no son numÃ©ricas: "
            f"{columnas_no_numericas}"
        )

    # --------------------------------------------------------
    # 2. Cobertura equivalente al nÃºmero de sigmas
    # --------------------------------------------------------

    # Para numero_sigmas = 3:
    #
    # cobertura_objetivo â‰ˆ 0.9973002
    #
    # Queda aproximadamente 0.00135 en cada cola.
    cobertura_objetivo = erf(
        numero_sigmas / sqrt(2)
    )

    probabilidad_cola = (
        1.0 - cobertura_objetivo
    ) / 2.0

    cuantil_inferior = probabilidad_cola
    cuantil_superior = 1.0 - probabilidad_cola

    print(
        f"Cobertura objetivo: "
        f"{cobertura_objetivo * 100:.4f}%"
    )

    print(
        f"Cuantil inferior: {cuantil_inferior:.6f}"
    )

    print(
        f"Cuantil superior: {cuantil_superior:.6f}"
    )

    print(
        f"Factor utilizado para IQR: {factor_iqr}"
    )

    # --------------------------------------------------------
    # 3. Crear la copia de trabajo
    # --------------------------------------------------------

    dataframe_normalizado = dataframe.copy()

    parametros_diccionario = {}
    filas_resumen = []

    # --------------------------------------------------------
    # 4. Procesar cada variable continua
    # --------------------------------------------------------

    for columna in columnas_continuas:

        serie = dataframe[columna].astype(float)
        valores_validos = serie.dropna()

        if valores_validos.empty:
            raise ValueError(
                f"La columna '{columna}' no contiene "
                "valores numÃ©ricos vÃ¡lidos."
            )

        # ----------------------------------------------------
        # 4.1. LÃ­mites obtenidos mediante cuantiles
        # ----------------------------------------------------

        limite_cuantil_inferior = (
            valores_validos.quantile(
                cuantil_inferior
            )
        )

        limite_cuantil_superior = (
            valores_validos.quantile(
                cuantil_superior
            )
        )

        # ----------------------------------------------------
        # 4.2. LÃ­mites obtenidos mediante IQR
        # ----------------------------------------------------

        q1 = valores_validos.quantile(0.25)
        q3 = valores_validos.quantile(0.75)

        iqr = q3 - q1

        if np.isclose(iqr, 0.0):
            raise ValueError(
                f"La columna '{columna}' tiene IQR igual a cero. "
                "No se puede aplicar el criterio hÃ­brido."
            )

        limite_iqr_inferior = (
            q1 - factor_iqr * iqr
        )

        limite_iqr_superior = (
            q3 + factor_iqr * iqr
        )

        # ----------------------------------------------------
        # 4.3. Seleccionar los lÃ­mites hÃ­bridos
        # ----------------------------------------------------

        # Para el lÃ­mite inferior escogemos el mÃ¡s alto.
        limite_inferior = max(
            limite_cuantil_inferior,
            limite_iqr_inferior
        )

        # Para el lÃ­mite superior escogemos el mÃ¡s bajo.
        limite_superior = min(
            limite_cuantil_superior,
            limite_iqr_superior
        )

        amplitud = (
            limite_superior
            - limite_inferior
        )

        if amplitud <= 0 or np.isclose(amplitud, 0.0):
            raise ValueError(
                f"Los lÃ­mites hÃ­bridos de '{columna}' "
                "no forman un intervalo vÃ¡lido."
            )

        # Identificar cuÃ¡l criterio determinÃ³ cada lÃ­mite.
        if np.isclose(
            limite_inferior,
            limite_cuantil_inferior
        ):
            criterio_inferior = "cuantil"
        else:
            criterio_inferior = "IQR"

        if np.isclose(
            limite_superior,
            limite_cuantil_superior
        ):
            criterio_superior = "cuantil"
        else:
            criterio_superior = "IQR"

        # ----------------------------------------------------
        # 5. Contar los valores que serÃ¡n recortados
        # ----------------------------------------------------

        cantidad_inferior = int(
            (serie < limite_inferior).sum()
        )

        cantidad_superior = int(
            (serie > limite_superior).sum()
        )

        total_recortados = (
            cantidad_inferior
            + cantidad_superior
        )

        porcentaje_recortado = (
            100
            * total_recortados
            / len(dataframe)
        )

        # ----------------------------------------------------
        # 6. Clipping
        # ----------------------------------------------------

        serie_recortada = serie.clip(
            lower=limite_inferior,
            upper=limite_superior
        )

        # ----------------------------------------------------
        # 7. NormalizaciÃ³n al intervalo [0, 1]
        # ----------------------------------------------------

        serie_normalizada = (
            serie_recortada
            - limite_inferior
        ) / amplitud

        dataframe_normalizado[columna] = (
            serie_normalizada
        )

        # ----------------------------------------------------
        # 8. Guardar parÃ¡metros
        # ----------------------------------------------------

        parametros_diccionario[columna] = {
            "numero_sigmas_equivalente":
                float(numero_sigmas),

            "cobertura_objetivo":
                float(cobertura_objetivo),

            "cuantil_inferior":
                float(cuantil_inferior),

            "cuantil_superior":
                float(cuantil_superior),

            "factor_iqr":
                float(factor_iqr),

            "q1":
                float(q1),

            "q3":
                float(q3),

            "iqr":
                float(iqr),

            "limite_cuantil_inferior":
                float(limite_cuantil_inferior),

            "limite_cuantil_superior":
                float(limite_cuantil_superior),

            "limite_iqr_inferior":
                float(limite_iqr_inferior),

            "limite_iqr_superior":
                float(limite_iqr_superior),

            "limite_inferior":
                float(limite_inferior),

            "limite_superior":
                float(limite_superior),

            "amplitud":
                float(amplitud),

            "criterio_inferior":
                criterio_inferior,

            "criterio_superior":
                criterio_superior,

            "cantidad_recortada_inferior":
                cantidad_inferior,

            "cantidad_recortada_superior":
                cantidad_superior
        }

        filas_resumen.append({
            "variable":
                columna,

            "q1":
                q1,

            "q3":
                q3,

            "IQR":
                iqr,

            "limite_cuantil_inf":
                limite_cuantil_inferior,

            "limite_IQR_inf":
                limite_iqr_inferior,

            "limite_inferior":
                limite_inferior,

            "criterio_inferior":
                criterio_inferior,

            "limite_cuantil_sup":
                limite_cuantil_superior,

            "limite_IQR_sup":
                limite_iqr_superior,

            "limite_superior":
                limite_superior,

            "criterio_superior":
                criterio_superior,

            "amplitud":
                amplitud,

            "recortados_inferior":
                cantidad_inferior,

            "recortados_superior":
                cantidad_superior,

            "total_recortados":
                total_recortados,

            "porcentaje_recortado":
                porcentaje_recortado
        })

    # --------------------------------------------------------
    # 9. Crear el resumen
    # --------------------------------------------------------

    resumen = pd.DataFrame(
        filas_resumen
    )

    return (
        dataframe_normalizado,
        parametros_diccionario,
        resumen
    )

In [ ]:
# 7. Aplicar la normalización robusta

columnas_continuas = [
    "edad_anios",
    "altura_cm",
    "peso_kg",
    "presion_sistolica",
    "presion_diastolica"
]

df_preparado_original = df_limpio.copy()

numero_sigmas = 3.0
factor_iqr = 3.0

(
    df_normalizado_robusto,
    parametros_normalizacion_robusta,
    resumen_normalizacion_robusta
) = normalizar_robustamente(
    dataframe=df_preparado_original,
    columnas_continuas=columnas_continuas,
    numero_sigmas=numero_sigmas,
    factor_iqr=factor_iqr
)

display(resumen_normalizacion_robusta.round(4))

verificacion_rangos = (
    df_normalizado_robusto[columnas_continuas]
    .agg(["min", "max", "mean", "std"])
    .T
)

display(verificacion_rangos.round(4))

if len(df_preparado_original) != len(df_normalizado_robusto):
    raise ValueError(
        "La normalización modificó el número de registros."
    )

for columna in columnas_continuas:
    minimo = df_normalizado_robusto[columna].min()
    maximo = df_normalizado_robusto[columna].max()

    if minimo < 0 or maximo > 1:
        raise ValueError(
            f"'{columna}' contiene valores fuera de [0, 1]."
        )

print(
    f"Registros conservados: {len(df_normalizado_robusto):,}"
)

In [ ]:
# 8. Metadatos de categorías

# Estos mapas se guardan para interpretar posteriormente las reglas.
# No realizan discretización ni binarización; los códigos originales
# se conservan en el dataset destinado a PSO.

mapas_categorias = {
    "sexo": {
        1: "mujer",
        2: "hombre"
    },
    "colesterol": {
        1: "normal",
        2: "arriba_de_lo_normal",
        3: "muy_arriba_de_lo_normal"
    },
    "glucosa": {
        1: "normal",
        2: "arriba_de_lo_normal",
        3: "muy_arriba_de_lo_normal"
    },
    "fumador": {
        0: "no",
        1: "si"
    },
    "alcohol": {
        0: "no",
        1: "si"
    },
    "actividad_fisica": {
        0: "no",
        1: "si"
    },
    "enfermedad_cardiovascular": {
        0: "no",
        1: "si"
    }
}


## 9. Guardado de resultados

Se conservan por separado los datos originales preparados, los datos normalizados para PSO, el resumen de transformación y los metadatos necesarios para interpretar las reglas.

In [ ]:
# ============================================================
# GUARDAR DATASETS Y PARÁMETROS EN REPOSITORIO LOCAL
# ============================================================

import os
import joblib


# ------------------------------------------------------------
# 1. Localizar la raiz del repositorio
# ------------------------------------------------------------

from pathlib import Path


def localizar_raiz_repositorio():
    """Encuentra la carpeta que contiene notebooks/ del repositorio."""
    for candidato in [Path.cwd(), *Path.cwd().parents]:
        if (candidato / "notebooks").is_dir():
            return candidato
    return Path.cwd()


raiz_repositorio = localizar_raiz_repositorio()
carpeta_destino = raiz_repositorio / "artifacts" / "preprocesamiento"
carpeta_destino.mkdir(parents=True, exist_ok=True)


# ------------------------------------------------------------
# 3. Verificaciones antes de guardar
# ------------------------------------------------------------

if len(df_preparado_original) != len(df_normalizado_robusto):
    raise ValueError(
        "El dataset original y el normalizado "
        "no tienen el mismo número de filas."
    )


if list(df_preparado_original.columns) != list(
    df_normalizado_robusto.columns
):
    raise ValueError(
        "El dataset original y el normalizado "
        "no tienen las mismas columnas."
    )


# Verificar que las variables continuas estén entre 0 y 1.
for columna in columnas_continuas:

    minimo = df_normalizado_robusto[columna].min()
    maximo = df_normalizado_robusto[columna].max()

    if minimo < 0 or maximo > 1:
        raise ValueError(
            f"La columna '{columna}' contiene valores "
            "fuera del intervalo [0, 1]."
        )


print("Verificaciones completadas correctamente.")


# ------------------------------------------------------------
# 4. Definir nombres de los archivos
# ------------------------------------------------------------

ruta_original = os.path.join(
    carpeta_destino,
    "cardiovascular_preparado_original.csv"
)

ruta_normalizado = os.path.join(
    carpeta_destino,
    "cardiovascular_normalizado_robusto.csv"
)

ruta_resumen = os.path.join(
    carpeta_destino,
    "resumen_normalizacion_robusta.csv"
)

ruta_metadatos = os.path.join(
    carpeta_destino,
    "metadatos_normalizacion_robusta.pkl"
)


# ------------------------------------------------------------
# 5. Guardar el dataset en unidades originales
# ------------------------------------------------------------

df_preparado_original.to_csv(
    ruta_original,
    index=False
)


# ------------------------------------------------------------
# 6. Guardar el dataset que utilizará PSO
# ------------------------------------------------------------

df_normalizado_robusto.to_csv(
    ruta_normalizado,
    index=False
)


# ------------------------------------------------------------
# 7. Guardar la tabla resumen de la normalización
# ------------------------------------------------------------

resumen_normalizacion_robusta.to_csv(
    ruta_resumen,
    index=False
)


# ------------------------------------------------------------
# 8. Preparar los metadatos
# ------------------------------------------------------------

columnas_categoricas = [
    "sexo",
    "colesterol",
    "glucosa",
    "fumador",
    "alcohol",
    "actividad_fisica"
]

columna_objetivo = (
    "enfermedad_cardiovascular"
)


metadatos_normalizacion = {
    # Parámetros específicos de cada variable continua
    "parametros_normalizacion":
        parametros_normalizacion_robusta,

    # Diccionario para traducir códigos categóricos
    "mapas_categorias":
        mapas_categorias,

    # Organización del dataset
    "columnas_continuas":
        columnas_continuas,

    "columnas_categoricas":
        columnas_categoricas,

    "columna_objetivo":
        columna_objetivo,

    # Configuración aplicada
    "metodo":
        "cuantiles_empiricos_e_IQR",

    "numero_sigmas":
        numero_sigmas,

    "factor_iqr":
        factor_iqr,

    # Información general
    "numero_registros":
        len(df_normalizado_robusto),

    "columnas_dataset":
        list(df_normalizado_robusto.columns)
}


# ------------------------------------------------------------
# 9. Guardar los metadatos con joblib
# ------------------------------------------------------------

joblib.dump(
    metadatos_normalizacion,
    ruta_metadatos
)


# ------------------------------------------------------------
# 10. Mostrar confirmación
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("ARCHIVOS GUARDADOS CORRECTAMENTE")
print("=" * 75)

print(
    f"\nDataset original:\n{ruta_original}"
)

print(
    f"\nDataset normalizado para PSO:\n{ruta_normalizado}"
)

print(
    f"\nResumen de normalización:\n{ruta_resumen}"
)

print(
    f"\nMetadatos y mapas:\n{ruta_metadatos}"
)

print(
    f"\nNúmero de registros guardados: "
    f"{len(df_normalizado_robusto):,}"
)

print(
    f"Número de columnas guardadas: "
    f"{df_normalizado_robusto.shape[1]}"
)